# Notebook 02 — Data Ingestion

**AI Evolution Map · Data Pipeline**

---

## Purpose

Fetch a small representative sample from each selected data source and save the
raw results under `data/raw/`. Nothing is cleaned or transformed here.

**Idempotency rule**: If a raw file already exists, we skip the API call and load
from disk. This prevents duplicate requests on re-runs.

## What you need

- An internet connection (the API calls run from this notebook).
- Optional: set `OPENALEX_CONTACT_EMAIL` in a `.env` file in the project root
  to join the OpenAlex polite pool and get higher rate limits.

No API keys are required for OpenAlex, arXiv, or Hugging Face public metadata.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path so we can import from src/
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import logging
from datetime import datetime, timezone

# Load optional environment variables (e.g. OPENALEX_CONTACT_EMAIL)
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / '.env')
    print('Loaded .env file (if present).')
except ImportError:
    print('python-dotenv not installed — skipping .env load.')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s — %(message)s',
    datefmt='%H:%M:%S'
)

RAW_DIR = project_root / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data directory: {RAW_DIR}')

Loaded .env file (if present).
Raw data directory: C:\Users\MANAS\Documents\AI-Evolution-Map\data\raw


In [2]:
# ── Helper: Idempotent Save ───────────────────────────────────────────────────
#
# We save both the API payload AND a sidecar provenance file.
# If the data file already exists, we skip the API call and return False.
# This is important: it means you can re-run this notebook without making
# duplicate API calls or overwriting known-good raw data.

def save_raw(
    data: list | dict,
    provenance: dict,
    filename: str,
    overwrite: bool = False
) -> Path:
    """
    Save raw API data and its provenance sidecar to data/raw/.

    Files saved:
      data/raw/{filename}.json         — the raw records
      data/raw/{filename}.provenance.json — retrieval metadata

    Returns the path to the data file.
    """
    data_path = RAW_DIR / f'{filename}.json'
    prov_path = RAW_DIR / f'{filename}.provenance.json'

    if data_path.exists() and not overwrite:
        print(f'  ⏭  {filename}.json already exists — skipping download.')
        return data_path

    with open(data_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)

    with open(prov_path, 'w', encoding='utf-8') as f:
        json.dump(provenance, f, indent=2, ensure_ascii=False, default=str)

    print(f'  ✓  Saved {len(data) if isinstance(data, list) else 1} records → {data_path.name}')
    return data_path


def load_raw(filename: str) -> list | dict:
    """Load a previously saved raw file from data/raw/."""
    data_path = RAW_DIR / f'{filename}.json'
    if not data_path.exists():
        raise FileNotFoundError(f'Raw file not found: {data_path}')
    with open(data_path, encoding='utf-8') as f:
        return json.load(f)

## Ingestion 1: OpenAlex — AI/ML Papers (2018–2023)

In [3]:
from src.ingestion.openalex_client import fetch_ai_works

OPENALEX_FILE = 'openalex_ai_works_2018_2023'

if not (RAW_DIR / f'{OPENALEX_FILE}.json').exists():
    print('Fetching OpenAlex works...')
    # Machine Learning field ID = 11 in OpenAlex topics
    # We use a topic filter with field.id:11 for broader coverage than concept IDs
    records, prov = fetch_ai_works(
        year_from=2018,
        year_to=2023,
        per_page=25,
        max_pages=4,  # 4 × 25 = 100 records max — enough to validate the pipeline
    )
    save_raw(records, prov.to_dict(), OPENALEX_FILE)
else:
    print(f'OpenAlex data already cached: {OPENALEX_FILE}.json')
    records = load_raw(OPENALEX_FILE)

print(f'OpenAlex records loaded: {len(records)}')

OpenAlex data already cached: openalex_ai_works_2018_2023.json


OpenAlex records loaded: 25


## Ingestion 2: arXiv — cs.AI Preprints

In [4]:
from src.ingestion.arxiv_client import fetch_arxiv_papers

ARXIV_FILE = 'arxiv_cs_ai_recent_50'

if not (RAW_DIR / f'{ARXIV_FILE}.json').exists():
    print('Fetching arXiv cs.AI preprints...')
    # NOTE: arXiv requires ≥3 second delay between calls (handled inside fetch_arxiv_papers)
    arxiv_records, arxiv_prov = fetch_arxiv_papers(
        search_query='cat:cs.AI',
        max_results=50,
        sort_by='submittedDate',
        sort_order='descending'
    )
    save_raw(arxiv_records, arxiv_prov.to_dict(), ARXIV_FILE)
else:
    print(f'arXiv data already cached: {ARXIV_FILE}.json')
    arxiv_records = load_raw(ARXIV_FILE)

print(f'arXiv records loaded: {len(arxiv_records)}')

arXiv data already cached: arxiv_cs_ai_recent_50.json
arXiv records loaded: 25


## Ingestion 3: Hugging Face Hub — Top Text-Generation Models

In [5]:
from src.ingestion.huggingface_client import fetch_hf_models

HF_FILE = 'huggingface_text_generation_top50'

if not (RAW_DIR / f'{HF_FILE}.json').exists():
    print('Fetching Hugging Face Hub model metadata...')
    hf_records, hf_prov = fetch_hf_models(
        filter_tags=['text-generation'],
        sort='downloads',
        direction=-1,
        limit=50,
        full=True  # Include model card data for license and language fields
    )
    save_raw(hf_records, hf_prov.to_dict(), HF_FILE)
else:
    print(f'HF Hub data already cached: {HF_FILE}.json')
    hf_records = load_raw(HF_FILE)

print(f'HF Hub records loaded: {len(hf_records)}')

HF Hub data already cached: huggingface_text_generation_top50.json
HF Hub records loaded: 25


In [6]:
# ── Ingestion Summary ─────────────────────────────────────────────────────────
print('=== Ingestion Summary ===')
for name, data in [
    ('OpenAlex', records),
    ('arXiv', arxiv_records),
    ('Hugging Face', hf_records),
]:
    count = len(data) if isinstance(data, list) else 1
    print(f'{name}: {count} records')

print('\nRaw files saved in:', RAW_DIR)
raw_files = list(RAW_DIR.glob('*.json'))
for f in sorted(raw_files):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:50s}  {size_kb:7.1f} KB')

=== Ingestion Summary ===
OpenAlex: 25 records
arXiv: 25 records
Hugging Face: 25 records

Raw files saved in: C:\Users\MANAS\Documents\AI-Evolution-Map\data\raw
  arxiv_cs_ai_recent_50.json                             52.4 KB
  arxiv_cs_ai_recent_50.provenance.json                   0.4 KB
  arxiv_cs_ai_sample.json                                52.4 KB
  arxiv_cs_ai_sample.provenance.json                      0.4 KB
  huggingface_models_sample.json                         62.9 KB
  huggingface_models_sample.provenance.json               0.4 KB
  huggingface_text_generation_top50.json                 62.9 KB
  huggingface_text_generation_top50.provenance.json       0.4 KB
  openalex_ai_works_2018_2023.json                      356.0 KB
  openalex_ai_works_2018_2023.provenance.json             0.5 KB
  openalex_ai_works_sample.json                         356.0 KB
  openalex_ai_works_sample.provenance.json                0.5 KB
